In [1]:
import os
import torch

from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from yacs.config import CfgNode as CN

from data.dataset import make_dataset
from src.utils import clean_exp_savedir
from src.losses import supervised_loss
import argparse

In [2]:
source_train_loader, _, source_test_loader, target_test_loader = (
    make_dataset(
        source_dataset="office31_amazon",
        target_dataset="office31_dslr",
        img_size=384,
        train_bs=16,
        eval_bs=256,
        num_workers=16,
    )
)

In [4]:
import torch
import torch.nn as nn
import copy

from src.components.torch_nn import make_backbone, make_classifier_head
from src.components.visual_prompt import MultiHeadVisualPrompt

from src.utils import freeze_layers

class SingleModel(nn.Module):
    def __init__(
        self,
        backbone_type:str="vit_b_16",
        in_dim:int=768,
        hidden_dim:int=256,
        out_dim:int=31,
        imgsize:int=384,
        attribute_layers=[5,6,5,6],
        patch_size=[4,8,16,32],
        attribute_channels=3,
        dropout=[0.1,0.1,0.2,0.2],
        attr_net=["conv", "conv", "transformer", "transformer"],
        freeze_backbone=True
    ):
        super(SingleModel, self).__init__()
        self.backbone = make_backbone(backbone_type)
        self.backbone.fc = nn.Identity()
        if freeze_backbone:
            freeze_layers([self.backbone])
        
        self.visual_prompt = MultiHeadVisualPrompt(
            imgsize=imgsize, 
            layers=attribute_layers, 
            patch_size=patch_size, 
            channels=attribute_channels, 
            dropout=dropout, 
            attr_net=attr_net
        )
        self.classifier_head = make_classifier_head(
            in_dim=in_dim, 
            hidden_dim=hidden_dim, 
            out_dim=out_dim, 
            dropout=0.1, 
            type="class",
        )

    def forward(self, x: torch.Tensor, head_idx: list[int] | None=None):
        prompted_imgs = self.visual_prompt(x, head_idx)
        output_dict = {}
        for head, imgs in prompted_imgs.items():
            feat = self.backbone(imgs)
            logit = self.classifier_head(feat)
            output_dict[head] = {}
            output_dict[head]['feat'] = feat
            output_dict[head]['logit'] = logit
        return output_dict

class ModelEMA:
    """ Model Exponential Moving Average """
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        # Disable gradient tracking for the EMA model
        for param in self.ema.parameters():
            param.requires_grad_(False)

    def update(self, model):
        # Update EMA parameters
        with torch.no_grad():
            for ema_v, model_v in zip(self.ema.state_dict().values(), model.state_dict().values()):
                if ema_v.dtype.is_floating_point:
                    ema_v.copy_(ema_v * self.decay + (1. - self.decay) * model_v)

model = SingleModel(
    backbone_type="vit_b_32", 
    attribute_layers=[5,6,5,6],
    patch_size=[8,32,16,24]
)
device = torch.device("cuda")
model = model.to(device)
ema_model = ModelEMA(model, decay=0.9996)

Loaded pretrained weights.


In [5]:
scaler = GradScaler('cuda')
optimizer = torch.optim.AdamW(
    [
        {
            "params": list(model.classifier_head.parameters()),
            "lr": 1e-3,
            "weight_decay": 1e-4,
        },
        {
            "params": list(model.visual_prompt.parameters()),
            "lr": 5e-4,
            "weight_decay": 1e-5,
        },
    ]
)

epochs = 30
total_steps = epochs * len(source_train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

In [6]:
import torch
import torch.nn as nn

@torch.no_grad() 
def evaluate(model, branch, test_loader, device, criterion=nn.CrossEntropyLoss()):
    model.eval() 
    head_correct = {}
    head_loss = {}
    total_samples = 0
    
    for batch_data in test_loader:
        img, labels = batch_data
        img = img.to(device)
        labels = labels.to(device)
        batch_size = labels.size(0)
        total_samples += batch_size
        
        with torch.amp.autocast('cuda'): 
            output_dict = model(img)
            
        for head_id, sub_dict in output_dict.items():
            logits = sub_dict['logit']
            loss = criterion(logits, labels)
            
            # Calculate predictions
            _, preds = torch.max(logits, 1)
            correct = (preds == labels).sum().item()
            
            # Initialize dictionaries for new heads dynamically
            if head_id not in head_correct:
                head_correct[head_id] = 0
                head_loss[head_id] = 0.0
                
            head_correct[head_id] += correct
            # Multiply loss by batch size to get the true running sum
            head_loss[head_id] += loss.item() * batch_size 
            
    # Calculate and print final per-head metrics
    avg_total_loss = 0.0
    avg_total_acc = 0.0
    num_heads = len(head_correct)
    
    print(f"\n--- Detailed Evaluation ({branch} branch) ---")
    for head_id in head_correct.keys():
        h_acc = (head_correct[head_id] / total_samples) * 100
        h_loss = head_loss[head_id] / total_samples
        
        print(f"  Head {head_id} | Loss: {h_loss:.4f} | Accuracy: {h_acc:.2f}%")
        
        avg_total_loss += h_loss
        avg_total_acc += h_acc
        
    avg_total_loss /= num_heads
    avg_total_acc /= num_heads
    
    # Return averages to satisfy your training loop's expectation of two return values
    return avg_total_loss, avg_total_acc

In [8]:
os.makedirs("exp", exist_ok=True)
exp_save_dir = os.path.join("exp", "exp_1")
best_test_acc = 0
# Training loop
for epoch in range(epochs):
    running_loss = 0.0
    model.train()
    pbar = tqdm(
        source_train_loader,
        total=len(source_train_loader),
        desc=f"Epoch {epoch + 1}",
        ncols=100,
    )

    for batch_idx, source_data in enumerate(pbar):
        pbar.set_description_str(f"Epoch {epoch + 1}", refresh=True)
        current_step = epoch * len(source_train_loader) + batch_idx
        # weak_img, strong_img, label
        _, strong_img, src_labels = source_data 

        strong_img = strong_img.to(device)
        src_labels = src_labels.to(device)
        optimizer.zero_grad()
        loss = 0.0
        with autocast('cuda'):
            output_dict = model(strong_img)
            for head_id, sub_dict in output_dict.items():
                loss += supervised_loss(sub_dict['logit'], src_labels)
            running_loss += loss.item()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema_model.update(model)

    test_loss_src, test_accuracy_src = evaluate(
        ema_model.ema, branch="src", test_loader=source_test_loader, device=device
    )
    test_loss_tgt, test_accuracy_tgt = evaluate(
         ema_model.ema, branch="tgt", test_loader=target_test_loader, device=device
    )
    
    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Source: {test_loss_src:.4f}, Test Accuracy Source: {test_accuracy_src:.2f}%"
    )
    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Target: {test_loss_tgt:.4f}, Test Accuracy Target: {test_accuracy_tgt:.2f}%"
    )

    # if test_accuracy_src > best_test_acc:
    #     best_test_acc = test_accuracy_src
    #     ckpt_path = os.path.join(
    #         exp_save_dir, f"bi_best_{test_accuracy_src:.2f}.pth"
    #     )
    #     torch.save(
    #         {
    #             "epoch": epoch,
    #             "best_test_acc": best_test_acc,
    #             "model_state_dict": model.state_dict(),
    #             "optimizer_state_dict": optimizer.state_dict(),
    #             "scaler_state_dict": scaler.state_dict(),
    #         },
    #         ckpt_path,
    #     )
    #     print(f"New best checkpoint saved: {ckpt_path}")
    #     if test_accuracy_src == 100:
    #         break

Epoch 1: 100%|████████████████████████████████████████████████████| 176/176 [00:41<00:00,  4.25it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.5929 | Accuracy: 96.49%
  Head conv_1 | Loss: 0.5929 | Accuracy: 96.59%
  Head transformer_2 | Loss: 0.5580 | Accuracy: 96.77%
  Head transformer_3 | Loss: 0.5629 | Accuracy: 96.77%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.9953 | Accuracy: 87.15%
  Head conv_1 | Loss: 0.9937 | Accuracy: 87.35%
  Head transformer_2 | Loss: 0.9537 | Accuracy: 87.95%
  Head transformer_3 | Loss: 0.9591 | Accuracy: 88.15%
Epoch [1/30] Test Loss Source: 0.5767, Test Accuracy Source: 96.65%
Epoch [1/30] Test Loss Target: 0.9754, Test Accuracy Target: 87.65%


Epoch 2: 100%|████████████████████████████████████████████████████| 176/176 [00:41<00:00,  4.28it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.4788 | Accuracy: 97.09%
  Head conv_1 | Loss: 0.4791 | Accuracy: 97.16%
  Head transformer_2 | Loss: 0.4462 | Accuracy: 97.09%
  Head transformer_3 | Loss: 0.4504 | Accuracy: 97.16%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.8798 | Accuracy: 87.15%
  Head conv_1 | Loss: 0.8789 | Accuracy: 87.35%
  Head transformer_2 | Loss: 0.8401 | Accuracy: 88.35%
  Head transformer_3 | Loss: 0.8463 | Accuracy: 88.15%
Epoch [2/30] Test Loss Source: 0.4636, Test Accuracy Source: 97.12%
Epoch [2/30] Test Loss Target: 0.8613, Test Accuracy Target: 87.75%


Epoch 3: 100%|████████████████████████████████████████████████████| 176/176 [00:40<00:00,  4.30it/s]


...


--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.0265 | Accuracy: 99.43%
  Head conv_1 | Loss: 0.0263 | Accuracy: 99.40%
  Head transformer_2 | Loss: 0.0261 | Accuracy: 99.43%
  Head transformer_3 | Loss: 0.0260 | Accuracy: 99.47%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.4352 | Accuracy: 86.55%
  Head conv_1 | Loss: 0.4410 | Accuracy: 86.95%
  Head transformer_2 | Loss: 0.4390 | Accuracy: 87.15%
  Head transformer_3 | Loss: 0.4370 | Accuracy: 86.95%
Epoch [28/30] Test Loss Source: 0.0262, Test Accuracy Source: 99.43%
Epoch [28/30] Test Loss Target: 0.4380, Test Accuracy Target: 86.90%


Epoch 29: 100%|███████████████████████████████████████████████████| 176/176 [00:41<00:00,  4.29it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.0254 | Accuracy: 99.43%
  Head conv_1 | Loss: 0.0252 | Accuracy: 99.50%
  Head transformer_2 | Loss: 0.0252 | Accuracy: 99.43%
  Head transformer_3 | Loss: 0.0250 | Accuracy: 99.47%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.4375 | Accuracy: 86.75%
  Head conv_1 | Loss: 0.4431 | Accuracy: 86.75%
  Head transformer_2 | Loss: 0.4422 | Accuracy: 87.15%
  Head transformer_3 | Loss: 0.4404 | Accuracy: 86.75%
Epoch [29/30] Test Loss Source: 0.0252, Test Accuracy Source: 99.46%
Epoch [29/30] Test Loss Target: 0.4408, Test Accuracy Target: 86.85%


Epoch 30: 100%|███████████████████████████████████████████████████| 176/176 [00:40<00:00,  4.31it/s]



--- Detailed Evaluation (src branch) ---
  Head conv_0 | Loss: 0.0244 | Accuracy: 99.43%
  Head conv_1 | Loss: 0.0243 | Accuracy: 99.47%
  Head transformer_2 | Loss: 0.0243 | Accuracy: 99.43%
  Head transformer_3 | Loss: 0.0241 | Accuracy: 99.50%

--- Detailed Evaluation (tgt branch) ---
  Head conv_0 | Loss: 0.4401 | Accuracy: 86.75%
  Head conv_1 | Loss: 0.4456 | Accuracy: 86.75%
  Head transformer_2 | Loss: 0.4461 | Accuracy: 86.95%
  Head transformer_3 | Loss: 0.4446 | Accuracy: 86.75%
Epoch [30/30] Test Loss Source: 0.0243, Test Accuracy Source: 99.46%
Epoch [30/30] Test Loss Target: 0.4441, Test Accuracy Target: 86.80%


In [23]:
def evaluate_class_wise(model, head_id,head_name, test_loader, device, num_classes):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()

    correct_per_class = torch.zeros(num_classes, device=device)
    total_per_class = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            pred = model(images, head_id)[head_name]['logit']
            loss = criterion(pred, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(pred, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Update class-wise metrics for the current batch
            for c in range(num_classes):
                class_mask = (labels == c)
                total_per_class[c] += class_mask.sum()
                correct_per_class[c] += (predicted[class_mask] == labels[class_mask]).sum()

    avg_loss = total_loss / total
    accuracy = 100 * correct / total
    
    # Calculate class-wise accuracy (in percentage) and handle division by zero
    class_wise_accuracy = torch.where(
        total_per_class > 0, 
        (correct_per_class / total_per_class) * 100, 
        torch.tensor(0.0, device=device)
    )

    return avg_loss, accuracy, class_wise_accuracy

In [24]:
evaluate_class_wise(ema_model.ema,  head_id=0,head_name = "conv_0", test_loader=target_test_loader, device=device, num_classes=31)

(0.44008335566903695,
 86.74698795180723,
 tensor([100.0000, 100.0000, 100.0000, 100.0000,  31.2500,  91.6667, 100.0000,
         100.0000,  80.0000, 100.0000, 100.0000,  80.0000,  79.1667,  87.5000,
          74.1936,  77.2727, 100.0000, 100.0000,  90.0000, 100.0000, 100.0000,
         100.0000, 100.0000,  83.3333,  20.0000, 100.0000, 100.0000,  65.3846,
          66.6667,  95.4545, 100.0000], device='cuda:0'))

In [25]:
evaluate_class_wise(ema_model.ema,  head_id=1,head_name = "conv_1", test_loader=target_test_loader, device=device, num_classes=31)

(0.4455284406861148,
 86.74698795180723,
 tensor([100.0000, 100.0000, 100.0000, 100.0000,  31.2500,  91.6667, 100.0000,
         100.0000,  80.0000, 100.0000, 100.0000,  80.0000,  79.1667,  87.5000,
          77.4193,  77.2727, 100.0000, 100.0000,  90.0000, 100.0000, 100.0000,
         100.0000, 100.0000,  83.3333,  20.0000, 100.0000, 100.0000,  57.6923,
          76.1905,  90.9091, 100.0000], device='cuda:0'))

In [27]:
evaluate_class_wise(ema_model.ema,  head_id=2,head_name = "transformer_2", test_loader=target_test_loader, device=device, num_classes=31)

(0.44624764277753104,
 86.94779116465864,
 tensor([100.0000, 100.0000, 100.0000, 100.0000,  37.5000,  91.6667, 100.0000,
         100.0000,  73.3333, 100.0000, 100.0000,  80.0000,  83.3333,  93.7500,
          67.7419,  81.8182, 100.0000, 100.0000,  90.0000, 100.0000, 100.0000,
         100.0000, 100.0000,  94.4444,  40.0000, 100.0000, 100.0000,  50.0000,
          71.4286,  90.9091, 100.0000], device='cuda:0'))

In [28]:
evaluate_class_wise(ema_model.ema,  head_id=3,head_name = "transformer_3", test_loader=target_test_loader, device=device, num_classes=31)

(0.44461566760358084,
 86.74698795180723,
 tensor([100.0000, 100.0000, 100.0000, 100.0000,  37.5000,  91.6667, 100.0000,
         100.0000,  80.0000, 100.0000, 100.0000,  80.0000,  79.1667,  93.7500,
          70.9677,  81.8182, 100.0000, 100.0000,  90.0000, 100.0000, 100.0000,
         100.0000,  95.6522,  88.8889,  40.0000, 100.0000, 100.0000,  53.8462,
          66.6667,  90.9091, 100.0000], device='cuda:0'))